# MatGraph-CLI Tutorial

Welcome to the official tutorial for **MatGraph-CLI**! In this notebook, we will explore how to use the MatGraph Python SDK for material property prediction, inverse design, and structural relaxation.

## 1. Installation
First, let's install the `matgraph-cli` package.

In [ ]:
!pip install matgraph-cli

## 2. Setup API Key
To access crystal structures from the Materials Project, you need an API key. You can get one from [materialsproject.org](https://materialsproject.org).

Let's set it as an environment variable.

In [ ]:
import os
# Replace with your actual Materials Project API Key
os.environ["MP_API_KEY"] = "YOUR_API_KEY"

from matgraph.sdk import MatGraphSDK
sdk = MatGraphSDK()

## 3. Property Prediction
Let's predict the properties of a material, for example, `Si` (Silicon) using the `cgcnn` model.

In [ ]:
results = sdk.predict("Si", model="cgcnn")
for r in results:
    print(f"Material ID: {r['material_id']}")
    print(f"Predicted Band Gap: {r['predicted_band_gap']:.3f} eV")
    print(f"Predicted Formation Energy: {r['predicted_form_energy']:.3f} eV/atom")
    print("-" * 30)

## 4. Inverse Design
We can search for materials that meet specific criteria. Let's find Cubic materials with a band gap between 1.0 and 2.0 eV, excluding Pb and Cd.

In [ ]:
design_results = sdk.design(
    min_gap=1.0,
    max_gap=2.0,
    crystal_system="Cubic",
    exclude_elements=["Pb", "Cd"],
    limit=5
)

for r in design_results:
    print(f"{r['formula']} | Gap: {r['band_gap']} eV | Stable: {r['is_stable']}")

## 5. Structural Relaxation (M3GNet + ASE)
We can perform structural relaxation using the M3GNet Universal Potential wrapped in an ASE Calculator.

In [ ]:
relax_results = sdk.relax("Si", steps=10)
print(f"Relaxation for {relax_results['formula']}")
print(f"Initial Energy: {relax_results['initial_energy']:.4f} eV")
print(f"Final Energy: {relax_results['final_energy']:.4f} eV")
print(f"Steps taken: {relax_results['steps_taken']}")

## 6. Model Evaluation
Evaluate a model's predicted properties against the true properties retrieved from the Materials Project.

In [ ]:
eval_results = sdk.evaluate("LiFePO4", model="cgcnn")
for r in eval_results:
    print(f"Material ID: {r['material_id']}")
    print(f"True Gap: {r['true_band_gap']:.3f} eV, Predicted Gap: {r['predicted_band_gap']:.3f} eV")
    print(f"True E_form: {r['true_form_energy']:.3f} eV/atom, Predicted: {r['predicted_form_energy']:.3f} eV/atom")
    print("-" * 40)

## 7. Generative Substitution
Substitute one element for another in a known structure, and evaluate the thermodynamic stability of the new hypothetical material using the true M3GNet potential.

In [ ]:
sub_results = sdk.substitute(formula="LiFePO4", elem_out="Li", elem_in="Na")
print("Original Material:", sub_results['original'])
print("Hypothetical Material:", sub_results['hypothetical'])
print("Is it more stable?", "Yes" if sub_results['is_more_stable'] else "No")

## 8. XRD Pattern Simulation
Simulate the X-Ray Diffraction (XRD) pattern for a material using CuKa radiation.

In [ ]:
xrd_data = sdk.xrd("Si")
import matplotlib.pyplot as plt

two_theta = xrd_data["two_theta"]
intensity = xrd_data["intensity"]

plt.figure(figsize=(10, 4))
plt.vlines(two_theta, 0, intensity, colors='b', lw=2)
plt.xlabel("2θ (degrees)")
plt.ylabel("Intensity (a.u.)")
plt.title("Simulated XRD Pattern for Silicon")
plt.show()

## 9. Phonon Density of States (DOS)
Fetch and visualize the Phonon DOS for a specific material.

In [ ]:
phonon_data = sdk.phonon("Si")

freqs = phonon_data["frequencies"]
densities = phonon_data["densities"]

plt.figure(figsize=(10, 4))
plt.plot(freqs, densities, color='r')
plt.fill_between(freqs, densities, color='r', alpha=0.3)
plt.xlabel("Frequency (THz)")
plt.ylabel("Density of States")
plt.title("Phonon DOS for Silicon")
plt.show()

## Conclusion
You have successfully used MatGraph-CLI for prediction, inverse design, and structural relaxation. Explore more features like Generative Substitution and XRD simulation in the documentation!